## Cellule Modeles

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor, 
                              AdaBoostRegressor, ExtraTreesRegressor, 
                              HistGradientBoostingRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

df = pd.read_csv('cleaned_dataset.csv', sep=';')

if 'Mois' not in df.columns:
    df['Date'] = df['Date'].astype(str).str.replace('/', '-').str.replace(' ', '-')
    df['Date'] = pd.to_datetime(df['Date'], format='mixed', errors='coerce')
    df['Mois'] = df['Date'].dt.month

df = df.dropna(subset=['Mois', 'Durée moyenne du trajet', "Retard moyen de tous les trains à l'arrivée"])

le_dep = LabelEncoder()
le_arr = LabelEncoder()

df['Gare_depart_enc'] = le_dep.fit_transform(df['Gare de départ'])
df['Gare_arrivee_enc'] = le_arr.fit_transform(df["Gare d'arrivée"])

X = df[['Gare_depart_enc', 'Gare_arrivee_enc', 'Mois', 'Durée moyenne du trajet']]
y = df["Retard moyen de tous les trains à l'arrivée"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


modeles = {
    "Regression Lineaire": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "SVR": SVR(kernel='rbf'),
    "Arbre de Decision": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Extra Trees": ExtraTreesRegressor(n_estimators=100, random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
    "MLP": MLPRegressor(hidden_layer_sizes=(50, 50), max_iter=500, random_state=42)
}

meilleur_nom = ""
meilleur_score = -float('inf')
meilleur_modele = None

for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    score = modele.score(X_test, y_test)
    print(f"{nom} -> Score R2 : {score:.4f}")
    
    if score > meilleur_score:
        meilleur_score = score
        meilleur_nom = nom
        meilleur_modele = modele

print(f"\nGagnant : {meilleur_nom} avec R2 = {meilleur_score:.4f}")

joblib.dump(meilleur_modele, 'model.joblib')
joblib.dump(le_dep, 'le_dep.joblib')
joblib.dump(le_arr, 'le_arr.joblib')



Regression Lineaire -> Score R2 : 0.1090
Ridge -> Score R2 : 0.1090
Lasso -> Score R2 : 0.1021
ElasticNet -> Score R2 : 0.1046
KNN -> Score R2 : 0.1710
SVR -> Score R2 : 0.1262
Arbre de Decision -> Score R2 : -0.0631
Random Forest -> Score R2 : 0.2181
Extra Trees -> Score R2 : 0.1174
AdaBoost -> Score R2 : -0.6527
Gradient Boosting -> Score R2 : 0.2795
Hist Gradient Boosting -> Score R2 : 0.2661
MLP -> Score R2 : 0.1046

Gagnant : Gradient Boosting avec R2 = 0.2795


['le_arr.joblib']